In [2]:
# =====================================================================
# Phase 4: Automated Hyperparameter Tuning with Optuna (XGBoost Edition)
# =====================================================================

import pandas as pd
import numpy as np
import xgboost as xgb
import optuna
import json
import os
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import r2_score, mean_squared_error

# Tắt bớt log rác của optuna để nhìn cho sạch
optuna.logging.set_verbosity(optuna.logging.WARNING)

# 1. Tải dữ liệu và chuẩn bị ma trận số học
df_train = pd.read_csv('../data/processed/train_features.csv')
df_test = pd.read_csv('../data/processed/test_features.csv')

metadata_cols = ['date', 'country_name', 'region', 'income_level', 'vector_disease_risk_score', 'target_log']
synthetic_leakage_cols = ['temperature_celsius', 'precipitation_mm', 'pm25_ugm3']
drop_cols = metadata_cols + synthetic_leakage_cols

features = [col for col in df_train.columns if col not in drop_cols]

X_train_fit = df_train[features].select_dtypes(include=[np.number])
y_train_log = df_train['target_log']
y_train_original = df_train['vector_disease_risk_score']

print(f"🚀 Ma trận sẵn sàng Tuning: {X_train_fit.shape}")

# 2. Định nghĩa hàm nghịch đảo log để tính Metric chuẩn
def calculate_cv_r2(y_true_original, y_pred_log):
    y_pred_original = np.expm1(y_pred_log)
    y_pred_original = np.clip(y_pred_original, 0, 100)
    return r2_score(y_true_original, y_pred_original)

# 3. Định nghĩa không gian tìm kiếm thông số (Objective Function)
def objective(trial):
    # Khai báo không gian thông số thông minh cho XGBoost
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 50, 300, step=50),
        'max_depth': trial.suggest_int('max_depth', 3, 9),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.2, log=True),
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
        'min_child_weight': trial.suggest_int('min_child_weight', 1, 10),
        'reg_alpha': trial.suggest_float('reg_alpha', 1e-3, 10.0, log=True),
        'reg_lambda': trial.suggest_float('reg_lambda', 1e-3, 10.0, log=True),
        'random_state': 42,
        'n_jobs': -1
    }
    
    tscv = TimeSeriesSplit(n_splits=5)
    cv_scores = []
    
    # Đánh giá bằng cơ chế TimeSeriesSplit nghiêm ngặt
    for train_idx, val_idx in tscv.split(X_train_fit):
        X_tr, y_tr = X_train_fit.iloc[train_idx], y_train_log.iloc[train_idx]
        X_va, y_va_orig = X_train_fit.iloc[val_idx], y_train_original.iloc[val_idx]
        
        model = xgb.XGBRegressor(**params)
        model.fit(X_tr, y_tr)
        
        preds_log = model.predict(X_va)
        r2 = calculate_cv_r2(y_va_orig, preds_log)
        cv_scores.append(r2)
        
    # Optuna sẽ tối ưu dựa trên trung bình điểm R2 của 5 tầng CV
    return np.mean(cv_scores)

# 4. Kích hoạt bộ não Bayesian Optimization
print("🔥 Đang khởi chạy Optuna để tìm kiếm bộ tham số tối ưu (Vui lòng đợi)...")
study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=30, timeout=600) # Thử nghiệm 30 kịch bản khác nhau trong max 10 phút

print("\n🏆 QUÁ TRÌNH TUNING HOÀN TẤT!")
print(f"-> Điểm Mean CV R² tốt nhất tìm được: {study.best_value:.4f}")
print("-> Bộ siêu tham số tối ưu:")
for k, v in study.best_params.items():
    print(f"   {k}: {v}")

# 5. Lưu trữ cấu hình tối ưu thành File cấu hình JSON phục vụ cho File 05
os.makedirs('../metrics', exist_ok=True)

result_to_save = study.best_params.copy()
result_to_save['best_score'] = study.best_value

with open('../metrics/best_params_xgboost.json', 'w') as f:
    json.dump(result_to_save, f, indent=4)

print("\n✔ Đã xuất file cấu hình tối ưu tại: '../metrics/best_params_xgboost.json'")

🚀 Ma trận sẵn sàng Tuning: (10350, 39)
🔥 Đang khởi chạy Optuna để tìm kiếm bộ tham số tối ưu (Vui lòng đợi)...

🏆 QUÁ TRÌNH TUNING HOÀN TẤT!
-> Điểm Mean CV R² tốt nhất tìm được: 0.7621
-> Bộ siêu tham số tối ưu:
   n_estimators: 300
   max_depth: 8
   learning_rate: 0.01799692090782518
   subsample: 0.9139560967506601
   colsample_bytree: 0.9999154600958405
   min_child_weight: 8
   reg_alpha: 0.13326768151923457
   reg_lambda: 0.43907601638665933

✔ Đã xuất file cấu hình tối ưu tại: '../metrics/best_params_xgboost.json'
